In [3]:
import ee
# ee.Authenticate()
import os
from google.cloud import storage
from google.oauth2 import service_account
# this is my workaround to get access to rpms data, leave this commented out if i forget!
SCOPES = ['https://www.googleapis.com/auth/earthengine']
credentials = service_account.Credentials.from_service_account_file(
    'fuelcast-storage-credentials.json',
    scopes=SCOPES
)
# ee.Initialize()
ee.Initialize(credentials=credentials)
poly_coords = {"coords":[]}
# import geemap
# Map = geemap.Map(center=[36,-120], zoom =8)


Bounding box drawing tool

In [14]:
from ipyleaflet import *
import numpy as np

zone_map = Map(center=(38, -97),
                zoom=5,
                basemap=basemap_to_tiles(basemaps.OpenStreetMap.Mapnik))

draw_control = DrawControl(
    rectangle= {
        "fillColor": "#fca45d",
        "color": "#fca45d",
        "fillOpacity": 0.2
    },
    polygon={},
    polyline={},
    circlemarker={}
    )

zone_map.add_control(draw_control)

def handle_draw(self, action, geo_json):
    poly_coords["coords"] = draw_control.last_draw['geometry']['coordinates'][0]
    print(poly_coords)
    print("Done generating coordinates.")
    return poly_coords
    
draw_control.on_draw(handle_draw)


zone_map

Map(center=[38, -97], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_te…

NameError: name 'poly_coords' is not defined

Make stack of LANDFIRE inputs:
    BPS code
    BPS name
    BPS model
    EVH
    EVC
    Lat/long

In [1]:
import ee
import pandas as pd
import numpy as np
import multiprocessing
from multiprocessing.dummy import Pool as ThreadPool
import time
from functools import partial
import logging
from datetime import datetime

def setup_logging(filename=None):
    if filename is None:
        filename = f"landfire_processing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    
    logging.basicConfig(
        filename=filename,
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    return filename

def pol_to_np(pol):
    """Converts list of coordinates to numpy array"""
    return np.array([list(l) for l in pol])

def pol_to_bounding_box(pol):
    """Converts list of coordinates to Earth Engine BBox"""
    arr = pol_to_np(pol)
    return ee.Geometry.BBox(
        np.min(arr[:,0]),
        np.min(arr[:,1]),
        np.max(arr[:,0]),
        np.max(arr[:,1])
    )

def get_landfire_stack():
    """Creates the combined image stack of all layers"""
    # BPS
    bps = ee.ImageCollection("LANDFIRE/Vegetation/BPS/v1_4_0").reduce(ee.Reducer.max())
    
    # EVH
    evh = ee.ImageCollection("projects/sat-io/open-datasets/landfire/vegetation/EVH").reduce(ee.Reducer.max())
    
    # RAP Cover
    rapcoverha = ee.Image("projects/rap-data-365417/assets/vegetation-cover-v3/2021").select('AFG')
    rapcoverhp = ee.Image("projects/rap-data-365417/assets/vegetation-cover-v3/2021").select('PFG')
    rapcovers = ee.Image("projects/rap-data-365417/assets/vegetation-cover-v3/2021").select('SHR')
    rapcoverh = rapcoverha.add(rapcoverhp)
    
    # SCLASS
    sclass = ee.ImageCollection("projects/sat-io/open-datasets/landfire/fire-regime/sclass").reduce(ee.Reducer.max())
    
    # RPMS
    rpms_col = ee.ImageCollection([])
    for year in range(2019, 2023):
        geotiff_path = f"gs://fuelcast-public/rpms/{year}/rpms_{year}.tif"
        image = ee.Image.loadGeoTIFF(geotiff_path)
        rpms_col = rpms_col.merge(ee.ImageCollection([image]))
    
    percentiles = [5, 25, 50, 75, 95]
    rpms_percentiles = rpms_col.reduce(ee.Reducer.percentile(percentiles))
    
    # NDVI
    ndvi_col = ee.ImageCollection("LANDSAT/COMPOSITES/C02/T1_L2_32DAY_NDVI")
    ndvi_years = range(2013, 2022)
    ndvi_collection = ee.ImageCollection.fromImages([
        ndvi_col.filterDate(f"{year}-01-01", f"{year}-12-31").median().set('year', year)
        for year in ndvi_years
    ])
    ndvi_percentiles = ndvi_collection.reduce(ee.Reducer.percentile(percentiles))
    
    # Precipitation
    gridmet = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET").select(['pr'])
    precip_years = range(1979, 2024)
    precip_collection = ee.ImageCollection.fromImages([
        gridmet.filterDate(f"{year}-01-01", f"{year}-12-31").sum().set('year', year)
        for year in precip_years
    ])
    precip_percentiles = precip_collection.reduce(ee.Reducer.percentile(percentiles))
    
    # Combine all layers
    return ee.Image([
        bps, rapcoverh, evh, ndvi_percentiles, precip_percentiles, 
        sclass, rpms_percentiles, rapcovers
    ]).toFloat()

def calculate_points_needed(chunk, scale=30):
    """Calculate number of points needed for 30m resolution in a chunk"""
    # Get chunk area in square meters
    area = chunk.area(30).getInfo()
    # Calculate points needed (one point per 30x30m pixel)
    return int(area / (scale * scale))

def split_geometry(geometry, max_points_per_chunk=5000):
    bounds = geometry.bounds()
    coords = bounds.coordinates().getInfo()[0]
    
    # Calculate area at 30m resolution
    total_area = geometry.area(30).getInfo()
    total_points = int(total_area / (30 * 30))
    
    # Calculate chunks needed to maintain 30m resolution
    num_chunks = max(int(np.ceil(total_points / max_points_per_chunk)), 1)
    num_splits = int(np.ceil(np.sqrt(num_chunks * 1.75)))  # Add 50% more splits for safety
    
    
    x_min = min(x[0] for x in coords)
    y_min = min(x[1] for x in coords)
    x_max = max(x[0] for x in coords)
    y_max = max(x[1] for x in coords)
    
    x_step = (x_max - x_min) / num_splits
    y_step = (y_max - y_min) / num_splits
    
    chunks = []
    for i in range(num_splits):
        for j in range(num_splits):
            chunk_x_min = x_min + (i * x_step)
            chunk_x_max = x_min + ((i + 1) * x_step)
            chunk_y_min = y_min + (j * y_step)
            chunk_y_max = y_min + ((j + 1) * y_step)
            
            chunk_geometry = ee.Geometry.Rectangle([
                chunk_x_min, chunk_y_min,
                chunk_x_max, chunk_y_max
            ])
            
            chunk = chunk_geometry.intersection(geometry, maxError=1)
            chunks.append(chunk)
    
    return chunks

def process_chunk(chunk, landfire_stack, scale=30):
    try:
        # Create a regular grid sample instead of random points
        samples = landfire_stack.sample(
            region=chunk,
            scale=scale,
            geometries=True,
            tileScale=4  # Add this to handle larger areas
        )
        
        data = samples.getInfo()
        logging.info(f"Successfully retrieved {len(data['features'])} features")
        
        rows = []
        for feature in data['features']:
            row_data = {
                'longitude': feature['geometry']['coordinates'][0],
                'latitude': feature['geometry']['coordinates'][1]
            }
            row_data.update(feature['properties'])
            rows.append(row_data)
        
        return pd.DataFrame(rows)
    
    except Exception as e:
        logging.error(f"Error processing chunk: {str(e)}")
        return pd.DataFrame()

def process_region(aoi, scale=30, max_points_per_chunk=10000, n_threads=4):
    """Process a region with parallel processing"""
    log_file = setup_logging()
    logging.info(f"Starting processing with {n_threads} threads")
    
    landfire_stack = get_landfire_stack()
    logging.info("Landfire stack created")
    
    if isinstance(aoi, ee.FeatureCollection):
        aoi = aoi.geometry()
    chunks = split_geometry(aoi, max_points_per_chunk)
    logging.info(f"Split geometry into {len(chunks)} chunks")
    
    process_func = partial(process_chunk, landfire_stack=landfire_stack, scale=scale)
    pool = ThreadPool(n_threads)
    
    try:
        chunk_dfs = pool.map(process_func, chunks)
        valid_dfs = [df for df in chunk_dfs if not df.empty]
        logging.info(f"Successfully processed {len(valid_dfs)} chunks")
        
        if valid_dfs:
            final_df = pd.concat(valid_dfs, ignore_index=True)
            logging.info(f"Final DataFrame shape: {final_df.shape}")
            cols = final_df.columns.tolist()
            cols.remove('longitude')
            cols.remove('latitude')
            final_df = final_df[['longitude', 'latitude'] + cols]
            return final_df
        else:
            logging.error("No valid data processed")
            return pd.DataFrame()
            
    except Exception as e:
        logging.error(f"Error in parallel processing: {str(e)}")
        return pd.DataFrame()
        
    finally:
        pool.close()
        pool.join()
        logging.info("Processing complete")



# Define your area of interest
if poly_coords["coords"] != []:
    print("Generating boundary with drawn geometry")
    aoi = pol_to_bounding_box(poly_coords["coords"])
else:
    print("Using manual override")
    # bounds = zone_ds.bounds
    aoi = ee.Geometry.BBox(-119.62601, 36.98082 , -119.12596, 37.23979)

# using cali boundary
use_cali = False
if use_cali == True:
    print("Using Cali bounds")
    boundary_geometry = 'users/colton11gerth/Cali'
    aoi = ee.FeatureCollection(boundary_geometry)

n_threads = max(1, multiprocessing.cpu_count() - 1)
# Process the region
start_time = time.time()
dfs_1 = process_region(aoi, scale=30, max_points_per_chunk=10000, n_threads=n_threads)
end_time = time.time()

print(f"Processing complete! Total time: {(end_time - start_time)/60:.2f} minutes")

print(f"DataFrame shape: {df.shape}")
logging.info(f"Total processing time: {(end_time - start_time)/60:.2f} minutes")
logging.info(f"DataFrame shape: {df.shape}")

# Save results
dfs_1.to_csv("landfire_pixel_data.csv", index=False)

NameError: name 'poly_coords' is not defined

If you already have your csv made, just load this cell

In [2]:
import ee
import pandas as pd
import numpy as np
import multiprocessing
from multiprocessing.dummy import Pool as ThreadPool
import time
from functools import partial
import logging
from datetime import datetime
dfs_1 = pd.read_csv("landfire_pixel_data.csv")

In [3]:
dfs_1.head()

,longitude,latitude,AFG,B0_p25,B0_p5,B0_p50,B0_p75,B0_p95,BPS_max,NDVI_p25,...,NDVI_p75,NDVI_p95,SHR,b1_max,b1_max_1,pr_p25,pr_p5,pr_p50,pr_p75,pr_p95
0,-119.625817,36.98099,73,3180.0,3087,3319.0,3912.5,4460,607,0.295453,...,0.309562,0.340551,4,304,7,327.849945,208.599991,403.947113,510.799988,709.024902
1,-119.625547,36.98099,70,3536.0,3394,3708.5,4604.5,5470,607,0.312712,...,0.329739,0.400061,5,304,7,327.849945,208.599991,403.947113,510.799988,709.024902
2,-119.625278,36.98099,73,3370.5,3329,3465.5,4242.5,4966,607,0.271446,...,0.298903,0.380625,2,304,7,327.849945,208.599991,403.947113,510.799988,709.024902
3,-119.625008,36.98099,73,3347.0,3101,3859.5,4220.0,4314,607,0.279315,...,0.312310,0.386024,2,304,7,327.849945,208.599991,403.947113,510.799988,709.024902
4,-119.624739,36.98099,72,3347.0,3101,3859.5,4220.0,4314,607,0.279315,...,0.312310,0.386024,3,304,7,327.849945,208.599991,403.947113,510.799988,709.024902


In [ ]:
from google.cloud import storage

bucket_id = "gigafire_rvs"

client = storage.Client()
bucket = client.get_bucket(bucket_id)
# list all objects in the directory
blobs = bucket.list_blobs(prefix="lf")
for blob in blobs:
    blob.delete()

#Export tiles to bucket (shard size 256)
def export_stack_to_cloud_storage(stack):
    # ee_region_bbox = ee_region.bounds()
    # print(ee_region_bbox.coordinates())
    task = ee.batch.Export.image.toCloudStorage(
        image=stack,
        fileNamePrefix="lf",
        bucket=bucket_id,
        scale=30,
        crs='EPSG:4326',
        shardSize=256,
        region=aoi,
        fileDimensions=256,
        skipEmptyTiles=True,
        maxPixels=1e13,
        fileFormat="GeoTIFF",
        formatOptions={
            'cloudOptimized': True
        }
    )
    task.start()

export_stack_to_cloud_storage(landfire_stack)


In [ ]:
###Download from bucket to tmp folder

from google.cloud import storage
import os
import tempfile

bucket_id = "gigafire_rvs"
#wd = "C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240226_test/temp"
# Create a temporary directory to download the files
temp_dir = tempfile.mkdtemp()
print(temp_dir)

def list_files_in_bucket(bucket_name):
    """List all files in a Google Cloud Storage bucket."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    files = []
    blobs = bucket.list_blobs(prefix="lf")
    for blob in blobs:
        files.append(blob.name)

    return files

def download_files_from_bucket(bucket_name, local_temp_dir):
    """Download files from a Google Cloud Storage bucket to a local directory."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)

    local_files = []
    blobs = bucket.list_blobs(prefix="lf")
    for blob in blobs:
        local_file_path = os.path.join(local_temp_dir, blob.name)
        blob.download_to_filename(local_file_path)
        local_files.append(local_file_path)

    return local_files




# Download files to temporary directory
local_files = download_files_from_bucket(bucket_id, temp_dir)
print("Downloaded files:", local_files)





In [ ]:
##Geotiff to dataframe

import rasterio
import pandas as pd




def multiband_geotiff_to_dataframe(geotiff_path):
    """Converts a multiband GeoTIFF file into a pandas DataFrame with x and y coordinates."""
    # Open the GeoTIFF file
    with rasterio.open(geotiff_path) as src:
        # Read all bands
        num_bands = src.count
        band_data = [src.read(i) for i in range(1, num_bands + 1)]
        
        # Get metadata
        transform = src.transform
        height, width = band_data[0].shape

        # Generate x and y coordinates
        x_coords = []
        y_coords = []
        for i in range(height):
            for j in range(width):
                x, y = src.xy(i, j)
                x_coords.append(x)
                y_coords.append(y)

        # Flatten band data and coordinates into DataFrame
        df_data = {'x': x_coords, 'y': y_coords}
        for band_idx in range(num_bands):
            band_name = src.descriptions[band_idx] if src.descriptions else f'band_{band_idx + 1}'
            df_data[band_name] = band_data[band_idx].flatten()

        df = pd.DataFrame(df_data)

    return df


# dfs_1 = [multiband_geotiff_to_dataframe(file_path) for file_path in local_files]

# for df in dfs_1:
#     # Identify NDVI columns
#     ndvi_columns = [col for col in df.columns if col.startswith('NDVI_')]
    
#     # Apply the power operation to NDVI columns
#     if ndvi_columns:  # Check if there are any NDVI columns
#         df[ndvi_columns] = df[ndvi_columns].apply(lambda x: x * 10000)


dfs_1[1].head(20)
# Clean up temporary directory
## for file_path in local_files:
    ##vos.remove(file_path)
## os.rmdir(temp_dir)




In [3]:
import pandas as pd
##Join df to kattribute table and reorder and clean columns
# key = pd.read_csv("C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240724_test/LF_140BPS_01122015.csv")
# height_key = pd.read_csv("C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240724_test/LF22_EVH_230.csv")
key = pd.read_csv("LF_140BPS_01122015.csv")
height_key = pd.read_csv("LF22_EVH_230.csv")

def join_new_attributes(df):  
    join = df.join(key[['VALUE','BPS_CODE', 'BPS_NAME','BPS_MODEL']].set_index('VALUE'), on='BPS_max')
    join = join.join(height_key[['VALUE','height']].set_index('VALUE'), on='b1_max', how="outer")
    join = join.assign(PLOT_ID=range(len(join)))
    cols = list(join.columns)
    Plots_Active = join[['PLOT_ID', 'BPS_CODE', 'BPS_NAME', 'BPS_MODEL', 
                     'AFG', 'height',
                     'latitude', 'longitude', *cols[10:15], *cols[5:10], 'b1_max_1', *cols[16:21]]]
    Plots_Active['BPS_CODE'] = Plots_Active['BPS_CODE'].fillna(0).astype(int)
    Plots_Active['b1_max_1'] = Plots_Active['b1_max_1'].fillna(0).astype(int)
    Plots_Active.dropna(subset=["BPS_MODEL"], inplace=True)
    Plots_Active['BPS_MODEL'] = Plots_Active['BPS_MODEL'].astype(str).str.zfill(7)
    
    return Plots_Active


dfs = join_new_attributes(dfs_1)
dfs.head
# dfs.to_csv("after_merge.csv")





C:\Users\colto\AppData\Local\Temp\ipykernel_6488\561770071.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Plots_Active['BPS_CODE'] = Plots_Active['BPS_CODE'].fillna(0).astype(int)
C:\Users\colto\AppData\Local\Temp\ipykernel_6488\561770071.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Plots_Active.dropna(subset=["BPS_MODEL"], inplace=True)
C:\Users\colto\AppData\Local\Temp\ipykernel_6488\561770071.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead



<bound method NDFrame.head of           PLOT_ID  BPS_CODE  \
0.0             0     10970   
1.0             1     10970   
2.0             2     10970   
3.0             3     10970   
4.0             4     10970   
...           ...       ...   
206317.0   219242     11050   
206395.0   219243     11050   
217080.0   219244     10322   
217233.0   219245     10310   
218409.0   219246     10321   

                                                   BPS_NAME BPS_MODEL   AFG  \
0.0                              California Mesic Chaparral   0610970  73.0   
1.0                              California Mesic Chaparral   0610970  70.0   
2.0                              California Mesic Chaparral   0610970  73.0   
3.0                              California Mesic Chaparral   0610970  73.0   
4.0                              California Mesic Chaparral   0610970  72.0   
...                                                     ...       ...   ...   
206317.0  Northern and Central California Dr

In [4]:
import numpy as np

#Join bps code to shrub cover to shrub info from combined growth rates table
# /key_shrubs = pd.read_csv("C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/rvs/BPS_Combined_Growthrates.csv")
key_shrubs = pd.read_csv("BPS_Combined_Growthrates.csv")
# key = pd.read_csv("C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240724_test/LF_140BPS_01122015.csv")
# height_key = pd.read_csv("C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240724_test/LF22_EVH_230.csv")
key = pd.read_csv("LF_140BPS_01122015.csv")
height_key = pd.read_csv("LF22_EVH_230.csv")
key['BPS_MODEL'] = key['BPS_MODEL'].replace(['na', 'nan'], np.nan)
key['BPS_MODEL'] = key['BPS_MODEL'].astype(float)
# species_key = pd.read_csv("C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240724_test/Bio_Crosswalk.csv")
species_key = pd.read_csv("Bio_Crosswalk.csv")
filtered_shrubs = key_shrubs[
    (key_shrubs['COMBINED_TYPE'].str.contains('S')) & 
    (key_shrubs['COVER_TYPE'] == 'Late') &
    (key_shrubs['ZONE'] == 6)]

#species_key.set_index('spp_code', inplace=True)
#species_key = species_key[~species_key.duplicated(keep='first')]

def make_shrubs_table(df):  
    join = df.join(key[['VALUE','BPS_CODE', 'BPS_NAME','BPS_MODEL']].set_index('VALUE'), on='BPS_max')
    join = join.join(height_key[['VALUE','height']].set_index('VALUE'), on='b1_max', how="outer")
    join = join.assign(PLOT_ID=range(len(join)))
    join = pd.merge(
        join,
        filtered_shrubs[['BPS_CODE', 'BPS_MODEL', 'Species_1','Species_2','Species_3','Species_4']],
     on=['BPS_MODEL', 'BPS_CODE'],
        how='left'
    )
    
    
    Shrubs_Active = join[['PLOT_ID', 'Species_1', 'Species_2','Species_3','Species_4','SHR', 'height', 'BPS_CODE', 'BPS_NAME']]
    Shrubs_Active['BPS_CODE'] = Shrubs_Active['BPS_CODE'].fillna(0).astype(int)
    
    
    # Insert new second column with all zeros (as string)
    Shrubs_Active.insert(1, 'Plot_Name', '0')
    lifeform_dict = species_key.set_index('spp_code')['lifeform'].to_dict()
    for col in ['Species_1', 'Species_2', 'Species_3', 'Species_4']:
        Shrubs_Active[col + '_Lifeform'] = Shrubs_Active[col].map(lifeform_dict)
    
    def find_first_shrub_species(row):
        if row['Species_1_Lifeform'] == 'Shrub':
            return row['Species_1']
        elif row['Species_2_Lifeform'] == 'Shrub':
            return row['Species_2']
        elif row['Species_3_Lifeform'] == 'Shrub':
            return row['Species_3']
        elif row['Species_4_Lifeform'] == 'Shrub':
            return row['Species_4']
        else:
            return pd.NA # Return NA if no shrub species is found

    Shrubs_Active['dom_spp'] = Shrubs_Active.apply(find_first_shrub_species, axis=1)
    

    Shrubs_Active['spp_code'] = Shrubs_Active['dom_spp']
 
    Shrubs_Active = Shrubs_Active[['PLOT_ID', 'Plot_Name', 'dom_spp', 'spp_code', 'SHR', 'height', 'BPS_CODE','BPS_NAME']]
    Shrubs_Active.rename(columns={'SHR': 'cover'}, inplace=True)
    

    Shrubs_Active = Shrubs_Active.dropna(subset=['spp_code'])
    
    
    return Shrubs_Active

 


dfs_shrubs = make_shrubs_table(dfs_1)
dfs_shrubs.head


C:\Users\colto\AppData\Local\Temp\ipykernel_6488\4022773005.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Shrubs_Active['BPS_CODE'] = Shrubs_Active['BPS_CODE'].fillna(0).astype(int)
C:\Users\colto\AppData\Local\Temp\ipykernel_6488\4022773005.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Shrubs_Active[col + '_Lifeform'] = Shrubs_Active[col].map(lifeform_dict)
C:\Users\colto\AppData\Local\Temp\ipykernel_6488\4022773005.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

<bound method NDFrame.head of         PLOT_ID Plot_Name dom_spp spp_code  cover  height  BPS_CODE  \
9             9         0    ADFA     ADFA    1.0    40.0     11050   
13           13         0    ADFA     ADFA    3.0    40.0     11050   
16           16         0    ADFA     ADFA    3.0    40.0     11050   
18           18         0    ADFA     ADFA    8.0    40.0     11050   
36           36         0    ADFA     ADFA    4.0    40.0     11050   
...         ...       ...     ...      ...    ...     ...       ...   
219235   219235         0    ADFA     ADFA    7.0     NaN     11050   
219236   219236         0    ADFA     ADFA   16.0     NaN     11050   
219238   219238         0    ADFA     ADFA   27.0     NaN     11050   
219242   219242         0    ADFA     ADFA   28.0     NaN     11050   
219243   219243         0    ADFA     ADFA   16.0     NaN     11050   

                                                 BPS_NAME  
9       Northern and Central California Dry-Mesic Chap...

In [5]:
import sqlite3
import pandas as pd
import os
import shutil
import time

# Copy template to home folder for reading and writing 
source_file = '../librvs/data/rvs_demo_fromRobbAddFieldTypeMissing.db'
counter=0


# n = dfs[0].shape[0]
# for i in dfs:
# counter=counter+1
destination_file = f'240724_test.db'

if os.path.exists(destination_file):
    os.remove(destination_file)

shutil.copy(source_file, destination_file)
print("Template DB copied successfully.")
database_path = destination_file
print(database_path)

# Create a connection to the database
conn = sqlite3.connect(database_path)

# Define the table name
table_name = "Plots_Active"

# Get the column names of the table
cursor = conn.cursor()
cursor.execute(f"PRAGMA table_info({table_name})")
cols = [column[1] for column in cursor.fetchall()]  + ['NPP_1', 'NPP_2', 'NPP_3', 'NPP_4', 'NPP_5']
print(cols)
# Read data from pandas DataFrame
# Assuming df is your pandas DataFrame with the same column names


dfs.columns=cols
# Replace values in the table with pandas DataFrame values
dfs.to_sql(table_name, conn, if_exists='replace', index=False)


cursor.execute(f"PRAGMA table_info({table_name})")
columns_info = cursor.fetchall()
npp_columns = [column[1] for column in columns_info if column[1].startswith("NPP")]

# Execute UPDATE statements to set NULL values to 0 for each NPP column
for column_name in npp_columns:
    update_query = f"UPDATE {table_name} SET {column_name} = 0 WHERE {column_name} IS NULL"
    cursor.execute(update_query)


table_name="Disturbance_Plots_NoFire"
cursor.execute(f"DROP TABLE {table_name}")

table_name="Dist_year7fire"
cursor.execute(f"DROP TABLE {table_name}")

#table_name="Disturbance_Plots"
# cursor.execute(f"DROP TABLE {table_name}")

print("DONE")
conn.commit()
conn.close()
    # Commit changes and close connection




Template DB copied successfully.
240724_test.db
['PLOT_ID', 'BPS_CODE', 'BPS_NAME', 'BPS_MODEL', 'herb_cover', 'herb_height', 'latitude', 'longitude', 'PPT_1', 'PPT_2', 'PPT_3', 'PPT_4', 'PPT_5', 'NDVI_1', 'NDVI_2', 'NDVI_3', 'NDVI_4', 'NDVI_5', 'sclass', 'NPP_1', 'NPP_2', 'NPP_3', 'NPP_4', 'NPP_5']
DONE


In [6]:
counter=0
n = dfs_shrubs.shape[1]
print()
print(n)

counter=counter+1
database_path = f'240724_test.db'


# Create a connection to the database
conn = sqlite3.connect(database_path)
table_name= 'Shrubs_Active'
# Get the column names of the table
cursor = conn.cursor()
cursor.execute(f"PRAGMA table_info({table_name})")
cols= [column[1] for column in cursor.fetchall()] 
print(cols)
cols = ['cover' if col == 'cover_o' else col for col in cols]
# Read data from pandas DataFrame
# Assuming df is your pandas DataFrame with the same column names
dfs_shrubs.columns=cols[0:n]
# Replace values in the table with pandas DataFrame values
dfs_shrubs.to_sql(table_name, conn, if_exists='replace', index=False) 
update_command = """
UPDATE Shrubs_Active
SET height = 100;
"""

# Execute the SQL command
cursor.execute(update_command)
print("Shrubs table added")


conn.commit()
conn.execute('VACUUM')
conn.close()
#Copy file from home to RVS git folder for RVS use
# source_file = f'240724_test.db'
# destination_file = f'240724_test.db'
# shutil.copy(source_file, destination_file)
# print("Final DB copied successfully to inputs")


8
['PLOT_ID', 'PLOT_NAME', 'dom_spp', 'spp_code', 'cover_o', 'height', 'BPS_CODE', 'BPS_NAME', 'height_ft', 'o_code', 'cover']
Shrubs table added


In [ ]:
# import pandas as pd
# import os
# import shutil
# import time
# import sqlite3

# rows = [50000]

# # DataFrame to store execution times


# # Number of iterations

# # DataFrame to store execution times
# execution_times_df = pd.DataFrame(columns=['Rows', 'Execution Time (seconds)'])

# for i in rows:
    
#     destination_file = f'C:/Users/ingli/Desktop/home/240625_shrubs_1.db'


#     conn = sqlite3.connect(destination_file)
#     cursor = conn.cursor()
    

#     cursor.execute(f'''
#     DELETE FROM Plots_Active
#     WHERE PLOT_ID NOT IN (
#         SELECT PLOT_ID FROM Plots_Active
#         ORDER BY PLOT_ID ASC
#         LIMIT {i}
#     )
#     ''')

#     cursor.execute(f'''
#     DELETE FROM Shrubs_Active
#     WHERE PLOT_ID >= {i}
#     ''')
        
#     conn.commit()
#     conn.execute('VACUUM')
#     # Close the database connection
#     conn.close()


#     source_file = f'C:/Users/ingli/Desktop/home/240625_shrubs_1.db'
#     destination_file = f'C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240226_test/240625_shrubs_500.db'
#     shutil.copy(source_file, destination_file)
#     print("File copied successfully.")



In [11]:
# Disturbances inputs
run_fire = True
fire_start = 5
fire_stop = 5
conn.close()

In [13]:
##ADD fire at YEAR 5 
import os
import sqlite3
import pandas as pd
import shutil

# files = os.listdir('240724_test/inputs')
# for i in range(1,33): 
database_path = f'240724_test.db'

# Create a connection to the database
conn = sqlite3.connect(database_path)

# Define the table name
    
n = dfs.shape[0]

if run_fire==False:
    table_name="Disturbance_Plots"
    cursor.execute(f"DROP TABLE {table_name}")

else:
    # Create a cursor object using the cursor() method
    cursor = conn.cursor()
    cursor.execute("DELETE FROM Disturbance_Plots")
    # Insert rows with PLOT_IDs from 1 to N
    for plot_id in range(0, n):
        insert_data_query = f'''
    INSERT OR REPLACE INTO Disturbance_Plots (PLOT_ID, PLOT_NAME, DIST_TYPE, DIST_SUBTYPE, P1_VAL, FREQ, START_YEAR, STOP_YEAR,P2_VAL, P3_VAL)
    VALUES ({plot_id}, '{plot_id}', 'FIRE', 'WILDFIRE', 100.0, 0, {fire_start}, {fire_stop},0,0)
    '''
        cursor.execute(insert_data_query)
    
conn.commit()
# Close the connection
conn.close()


# destination_file = f'C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240724_test/inputs/240724_{i}_fire.db'
# shutil.copy(database_path, destination_file)
# print("Final fire  DB copied successfully to inputs")

In [ ]:
import os
files = os.listdir('C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240724_test/inputs')

for i in range(1, len(files) + 1):
    command = f'docker run -v "C:/Users/ingli/OneDrive - University of Nevada, Reno/Desktop/unr/git/RVS/240724_test":"/data" rvs-image-2024-dev /bin/bash -c "rvs /data/inputs/240724_{i}.db /data/outs/out_{i}.db 10 true; cp RVS_Debug.txt /data/"'
    print(command)